# Gestructureerde data: preprocessing dataset

In dit voorbeeld gaan we verder werken op de titanic dataset bestudeerd in voorgaande notebook.
Hierbij gaan we preprocessing toevoegen of in meer detail bespreken.

Hierbij is een belangrijk onderscheid tussen pytorch en tensorflow op te merken. Hierbij is het belangrijk om te realiseren dat pytorch verwacht dat alle data dat uit de __get__item functie komt numeriek is. Deze moet niet noodzakelijk reeds geschaald zijn maar moet wel dat datatype hebben. Het is dus belangrijk om een goede keuze te maken op welke plaats de nodige preprocessing stappen uitgevoerd worden. 
In het geval van tensorflow (in het geval van mixed datatypes) plaatsen we van in het begin elke feature apart. Daarna voegen we gelijkaardige features samen om deze te preprocessen. Pas op het einde van dit proces worden alle features samengevoegd tot een numerieke tensor.

### Pytorch

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Load Titanic dataset
titanic = pd.read_csv("https://storage.googleapis.com/tf-datasets/titanic/train.csv")
display(titanic)

# Define the preprocessing pipeline
numeric_features = ['age', 'n_siblings_spouses', 'parch', 'fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['sex', 'class', 'embark_town', 'deck', 'alone']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Define custom PyTorch Dataset
class TitanicDataset(Dataset):
    def __init__(self, dataframe):
        # Separate features and target
        self.X = dataframe.drop('survived', axis=1)
        y = dataframe['survived']

        self.y = torch.tensor(y.values, dtype=torch.long)

    def train_preprocessor(self, subset):
        return preprocessor.fit(self.X)

    def preprocess(self, preprocessor):
        self.X = torch.tensor(preprocessor.transform(self.X), dtype=torch.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Train test
titanic_train = titanic.sample(frac=0.8)
titanic_test = titanic.drop(titanic_train.index)

# Create the dataset
dataset_train = TitanicDataset(titanic_train)
dataset_test = TitanicDataset(titanic_test)

# preprocess
preprocessor = dataset_train.train_preprocessor(preprocessor)
dataset_train.preprocess(preprocessor)
dataset_test.preprocess(preprocessor)

# Create dataloaders
train_loader = DataLoader(dataset_train, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset_test, batch_size=32, shuffle=False)

## Oefening: Inkomensdataset

Met onderstaande code downloaden we een csv met gegevens rond het inkomen van volwassenen.
Plaats in de code-cellen eronder de nodige code om de volgende preprocessingstappen uit te voeren met pytorch:

* Splits in train-test
* Label kolom is het inkomen -> voer hiervoor ordinal encoding uit
* Voer normalisatie uit van de numerieke kolommen zodat alle waarden tussen 0 en 1 liggen (geen standaardverdeling)
* Voer ordinal encoding uit op de occupation en native_country kolom
* Voer one-hot encoding uit op de overige niet-numerieke kolommen

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 
           'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 
           'hours_per_week', 'native_country', 'income']
adult_income = pd.read_csv(url, header=None, names=columns)

adult_income.head()

In [ ]:
# pytorch
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder
from torch.utils.data import Dataset
import torch

class AdultIncomeDataset(Dataset):
    def __init__(self, data, scaler=None, ordinal_enc=None, onehot_enc=None, train=True):
        columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 
                   'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 
                   'hours_per_week', 'native_country', 'income']
        
        # Load and set up columns
        self.data = pd.read_csv(data, header=None, names=columns)
        
        # Identify the columns
        self.numerical_cols = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
        self.ordinal_cols = ['occupation', 'native_country']
        self.onehot_cols = ['workclass', 'education', 'marital_status', 'relationship', 'race', 'sex']
        
        # Ordinal encoding for 'income' (target label)
        income_enc = OrdinalEncoder(categories=[[' <=50K', ' >50K']])
        self.data['income'] = income_enc.fit_transform(self.data[['income']])

        # Normalize numerical columns (MinMaxScaler)
        if scaler is None:
            self.scaler = MinMaxScaler()
            self.data[self.numerical_cols] = self.scaler.fit_transform(self.data[self.numerical_cols])
        else:
            self.scaler = scaler
            self.data[self.numerical_cols] = self.scaler.transform(self.data[self.numerical_cols])
        
        # Ordinal encoding for specific columns
        if ordinal_enc is None:
            self.ordinal_enc = OrdinalEncoder()
            self.data[self.ordinal_cols] = self.ordinal_enc.fit_transform(self.data[self.ordinal_cols])
        else:
            self.ordinal_enc = ordinal_enc
            self.data[self.ordinal_cols] = self.ordinal_enc.transform(self.data[self.ordinal_cols])
        
        # One-hot encoding for categorical columns
        if onehot_enc is None:
            self.onehot_enc = OneHotEncoder(sparse=False, drop='first')
            onehot_encoded = self.onehot_enc.fit_transform(self.data[self.onehot_cols])
        else:
            self.onehot_enc = onehot_enc
            onehot_encoded = self.onehot_enc.transform(self.data[self.onehot_cols])
        
        # Combine data
        self.data = pd.concat([self.data.drop(columns=self.onehot_cols), 
                               pd.DataFrame(onehot_encoded, index=self.data.index)], axis=1)
        
        # Separate features and labels
        self.X = self.data.drop(columns='income').values
        self.y = self.data['income'].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return X, y
